In [1]:
# import sys
# sys.path.insert(0,"./../") #so we can import our modules properly
#!pip install IPython
#!pip install matplotlib
#!pip install pandas
#!pip install h5py
#!pip install tables

In [2]:
!pip list

Package                 Version
----------------------- -----------
appnope                 0.1.4
asttokens               3.0.0
blosc2                  3.3.4
certifi                 2025.4.26
charset-normalizer      3.4.2
comm                    0.2.2
contourpy               1.3.2
cycler                  0.12.1
debugpy                 1.8.14
decorator               5.2.1
executing               2.2.0
fonttools               4.58.1
h5py                    3.13.0
idna                    3.10
ipykernel               6.29.5
ipython                 9.3.0
ipython_pygments_lexers 1.1.1
jedi                    0.19.2
jupyter_client          8.6.3
jupyter_core            5.8.1
kiwisolver              1.4.8
matplotlib              3.10.3
matplotlib-inline       0.1.7
msgpack                 1.1.0
ndindex                 1.10.0
nest-asyncio            1.6.0
numexpr                 2.10.2
numpy                   2.2.6
packaging               25.0
pandas                  2.3.0
parso                

In [3]:
## iPhython
#from IPython.core.display import HTML
##display(HTML("<style>.container { width:90% !important; }</style>"))
#HTML("<style>.container { width:90% !important; }</style>")

In [4]:
import os
import h5py
import numpy as np
import pandas as pd

from matplotlib import rcParams
import matplotlib.pyplot as plt
%matplotlib notebook

# House B 

In [9]:
#path_to_house = "./datasets/dfB_300s.hdf"
home = os.getcwd()
ddir = "../data/Data_vs01/data" # 5-min data
fn = os.path.join(ddir, "dfB_300s.hdf")
print(os.path.isfile(fn))


True


In [14]:
# Read back
with pd.HDFStore(fn) as store:
    df = store['data']

In [16]:
df.tail()

,B_batt_state,B_boilertemp_bottom,B_boilertemp_top,B_boiler_heater_1_on,B_boiler_heater_2_on,B_boiler_heater_3_on,B_boiler_on_thermostat,B_boiler_power,B_direct_cons_power,B_from_batt_power,B_from_net_power,B_pv_prod_power,B_total_cons_power,B_to_batt_power,B_to_net_power
2019-07-30 23:40:00,54.0,54.5,60.4,0.0,0.0,0.0,1.0,0.0,0.0,320.52,0.0,0.0,320.52,0.0,0.0
2019-07-30 23:45:00,53.0,54.4,60.5,0.0,0.0,0.0,1.0,0.0,0.0,302.40,0.0,0.0,302.40,0.0,0.0
2019-07-30 23:50:00,53.0,54.4,60.4,0.0,0.0,0.0,1.0,0.0,0.0,236.64,0.0,0.0,236.64,0.0,0.0
2019-07-30 23:55:00,53.0,54.3,60.4,0.0,0.0,0.0,1.0,0.0,0.0,223.56,0.0,0.0,223.56,0.0,0.0
2019-07-31 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Data set preparation

* filter out missing entries
* perform sanity checks
* save preprocessed data
* load preprocessed
* create cleaned data set


In [ ]:


meters = ['B_boiler_power']

# Investigate only points in time where all values are available
cols = ['B_total_cons_power']
cols.extend(meters)
df_house_noNAN = df.loc[:,cols].dropna(axis=0, how='any')

# Check if total consumed power is larger than total of appliances
delta = df_house_noNAN.loc[:,meters].sum(axis=1) - df_house_noNAN.loc[:,'B_total_cons_power']
tmt = delta > 0

if np.any(tmt):
    print('Boiler Power is larger than total consumed power for some data points.')

    # investigate problematic cases
    print("Found {} out of {} ({:.2}%) values to be problematic.".format(tmt.sum(), len(tmt), tmt.sum()/len(tmt)*100))
    print("")
    print("Statistics of problematic values:")
    print(delta[tmt].describe())
    print()
    print((delta[tmt]/df_house_noNAN.loc[tmt, 'B_total_cons_power']).describe())
else: 
    print("Total of all appliances is smaller than total consumed energy for all measurement points.")


In [ ]:
l = ['Total Appliances = Boiler', 'Total Consumed']
if np.any(tmt):
    print('Boiler Power is larger than total consumed power for some data points.')
    
    ## plot params ##
    ha = 6
    ncols = 8
    nrows = np.sum(tmt)//ncols+1
    fig, ax = plt.subplots(figsize=(17,3*nrows), ncols=ncols, nrows=nrows)
    idxs = np.nonzero(tmt)[0]
    for i, idx in enumerate(idxs):
        minidx = idx-ha
        maxidx = idx+ha
        df_house_noNAN.iloc[minidx:maxidx].loc[:,'B_boiler_power'].plot(ax=ax[i//ncols, i%ncols], label='Total Appliances')
        df_house_noNAN.iloc[minidx:maxidx].loc[:,'B_total_cons_power'].plot(ax=ax[i//ncols, i%ncols], label='Total Consumed')
    fig.legend(l, loc='upper center')
else: 
    print("Total of all appliances is smaller than total consumed energy for all measurement points.")

## Visualize Boiler Variables

In [ ]:
fig, ax = plt.subplots(ncols=1, nrows=3, figsize=(17,12), sharex=True)
df_house.loc[:,['B_boilertemp_top', 'B_boilertemp_bottom']].plot(ax=ax[0])
ax[0].legend()
ax2 = ax[0].twinx()
df_house.loc[:,'B_boiler_power'].plot(ax=ax2, color='green')
ax2.legend()
df_house.loc[:,['B_boiler_heater_1_on', 'B_boiler_heater_2_on', 'B_boiler_heater_3_on']].plot(ax=ax[1])
ax[1].legend()
df_house.loc[:,['B_boiler_on_thermostat']].plot(ax=ax[2])
ax[2].legend()

## Check that `B_pv_prod_power` is the sum of `B_to_batt_power` + `B_direct_cons_power`+ `B_to_net_power`

In [ ]:
# Investigate only points in time where all values are available
df_house_noNAN = df_house.loc[:,['B_pv_prod_power', 'B_to_batt_power', 'B_direct_cons_power', 'B_to_net_power']].dropna(axis=0, how='any')
delta = df_house_noNAN.loc[:,'B_pv_prod_power'] - df_house_noNAN.loc[:,['B_to_batt_power', 'B_direct_cons_power', 'B_to_net_power']].sum(axis=1)
print("Statistics of deviations:")
delta.describe()

## Check that `B_total_cons_power` is the sum of `B_direct_cons_power` + `B_from_batt_power` + `B_from_net_power`

In [ ]:
# Investigate only points in time where all values are available
df_house_noNAN = df_house.loc[:,['B_total_cons_power', 'B_direct_cons_power', 'B_from_batt_power', 'B_from_net_power']].dropna(axis=0, how='any')
delta = df_house_noNAN.loc[:,'B_total_cons_power'] - df_house_noNAN.loc[:,['B_direct_cons_power', 'B_from_batt_power', 'B_from_net_power']].sum(axis=1)
print("Statistics of deviations:")
delta.describe()

## Visualize Batter Power Flows

In [ ]:
fig, ax = plt.subplots(figsize=(17,4))
df_house.loc[:,['B_to_batt_power', 'B_from_batt_power']].plot(ax=ax)
ax.legend(loc='upper left')
ax2 = ax.twinx()
df_house.loc[:,'B_batt_state'].plot(ax=ax2, color='green')
ax2.legend(loc='lower left')

# House B - Raw Data

In [ ]:
import os
base_path = "../rawData/B/"
files = os.listdir(path=base_path)

blacklisted = ["capPeriods.hdf"]
for file in files:
    if file in blacklisted: continue
    print(file)
    print(os.path.join(base_path, file))
    df_rawData = pd.read_hdf(os.path.join(base_path, file))
    print(df_rawData.iloc[1,0]-df_rawData.iloc[0,0])
    print(df_rawData.iloc[-1,0]-df_rawData.iloc[-2,0])
    print()